# 🗑️ Trash Classifier — Model Training with ResNet-18

This notebook trains a **ResNet-18** model using **transfer learning** to classify trash images into **12 categories**.

### Strategy
1. Load a **pre-trained ResNet-18** (trained on ImageNet — 1.2M images, 1000 classes)
2. **Freeze** all convolutional layers (feature extractor)
3. **Replace** the final FC layer with a new 12-class output head
4. **Train** only the FC layer for 5 epochs
5. **Evaluate** on validation and test sets

### Results
| Metric | Accuracy |
|--------|----------|
| Train | 92.34% |
| Validation | 92.08% |
| Test | 92.14% |

> **Hardware:** Trained on Kaggle with an NVIDIA Tesla T4 GPU

---
## 1. Import Libraries

In [3]:
import torch                             # Core PyTorch library for tensor operations
from torchvision import transforms       # Image transformations (resize, augment, normalize)
from torch.utils.data import DataLoader  # Efficient batched data loading with shuffling
from torchvision import models           # Pre-trained model zoo (ResNet, VGG, etc.)
import torch.nn as nn                    # Neural network layers, loss functions

---
## 2. Download Dataset from Google Drive

The dataset is hosted on Google Drive as a zip file. We use `gdown` to download it directly into the Kaggle environment.

> **Note:** This cell is specific to the Kaggle/Colab environment. If running locally, place your `data/` folder manually.

In [4]:
# Install gdown — a utility to download files from Google Drive
!pip install gdown
import gdown

# Google Drive file ID extracted from the sharing link
file_id = '1UrTmhu2nLvWMv-AMfoa3Z6JBZwalagA7'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'  # Name for the downloaded file

# Download the dataset zip file (quiet=False shows progress bar)
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1UrTmhu2nLvWMv-AMfoa3Z6JBZwalagA7
To: /kaggle/working/data.zip
100%|██████████| 251M/251M [00:01<00:00, 215MB/s]  


'data.zip'

### 2.1 Extract the Dataset

Unzip the downloaded file into a working directory.

In [5]:
# Unzip the dataset
# -q  : quiet mode (suppresses the list of every file being extracted)
# -d  : specifies the destination directory
!unzip -q /kaggle/working/data.zip -d /kaggle/working/my_data

---
## 3. Define Dataset Paths

Point to the extracted `train/` and `val/` directories containing class-organized images.

> **Note:** Update these paths if running on a local machine instead of Kaggle.

In [6]:
# Paths to the pre-split training and validation image directories
# Each contains subfolders named by class (e.g., train/battery/, train/plastic/)
train_path = "/kaggle/working/my_data/content/drive/MyDrive/datasets/data/train"
val_path   = "/kaggle/working/my_data/content/drive/MyDrive/datasets/data/val"

---
## 4. Define Image Transforms

### Training Transforms (with Data Augmentation)
Data augmentation artificially increases the diversity of the training set by applying random transformations. This helps the model generalize better and reduces overfitting.

| Transform | Purpose |
|-----------|---------|
| `Resize(224, 224)` | ResNet-18 expects 224×224 input |
| `RandomHorizontalFlip` | Randomly mirror images (50% chance) |
| `RandomRotation(10)` | Rotate by up to ±10 degrees |
| `ColorJitter` | Vary brightness & contrast randomly |
| `ToTensor` | Convert PIL image → PyTorch tensor |
| `Normalize` | Standardize using ImageNet statistics |

### Validation/Test Transforms (No Augmentation)
Only resize and normalize — we want deterministic, unmodified images for fair evaluation.

In [7]:
# ── Training Transforms ──────────────────────────────────────
# Includes data augmentation to improve model generalization
train_transforms=transforms.Compose([
    transforms.Resize((224,224)),              # Resize to 224x224 (ResNet-18 input size)
    transforms.RandomHorizontalFlip(),         # Randomly flip images horizontally (50% chance)
    transforms.RandomRotation(10),             # Randomly rotate images by up to ±10 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Random brightness/contrast changes
    transforms.ToTensor(),                     # Convert PIL Image → Tensor (HWC → CHW, scale to [0,1])
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],            # ImageNet channel means (R, G, B)
        std=[0.229, 0.224, 0.225]              # ImageNet channel standard deviations
    )
])

# ── Validation / Test Transforms ─────────────────────────────
# No augmentation — only resize and normalize for consistent evaluation
val_test_transforms=transforms.Compose([
    transforms.Resize((224,224)),              # Resize to match training input size
    transforms.ToTensor(),                     # Convert to tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],            # Same ImageNet means as training
        std=[0.229, 0.224, 0.225]              # Same ImageNet stds as training
    )
])

---
## 5. Load Datasets with ImageFolder

`ImageFolder` automatically loads images from a directory structure where each subfolder is a class:
```
train/
├── battery/      → class 0
├── biological/   → class 1
├── brown-glass/  → class 2
└── ...           → class N
```
Labels are assigned as integers based on **sorted** folder names.

In [8]:
from torchvision import datasets

# Load training images and apply training transforms (with augmentation)
train_data=datasets.ImageFolder(train_path,transform=train_transforms)

# Load validation images and apply validation transforms (no augmentation)
val_data=datasets.ImageFolder(val_path,transform=val_test_transforms)

---
## 6. Create DataLoaders (Train / Validation / Test)

The physical `val/` folder is further split **50/50** into:
- **Validation set** — used to monitor performance during training
- **Test set** — used for final, unbiased evaluation

A fixed random seed (`42`) ensures the split is **reproducible**.

In [25]:
import torch
from torch.utils.data import random_split, DataLoader

# Training DataLoader — shuffle=True for random batch composition each epoch
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Split the validation folder 50/50 into validation and test sets
total_val_images = len(val_data)
test_size = total_val_images // 2             # Half for testing
new_val_size = total_val_images - test_size   # Remaining half for validation

# Use a fixed seed for reproducibility — same split every time
generator = torch.Generator().manual_seed(42) 
new_val_data, test_data = random_split(val_data, [new_val_size, test_size], generator=generator)

# Create separate DataLoaders for validation and test
# shuffle=False ensures consistent evaluation ordering
val_loader = DataLoader(new_val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Print the data distribution summary
print(f"Total Images in physical 'val' folder: {total_val_images}")
print(f"--> Assigned to Validation Loader: {len(new_val_data)}")
print(f"--> Assigned to Test Loader: {len(test_data)}")

Total Images in physical 'val' folder: 3106
--> Assigned to Validation Loader: 1553
--> Assigned to Test Loader: 1553


---
## 7. Custom CNN Architecture (Experimental — Not Used)

This was an initial attempt at building a CNN from scratch before switching to transfer learning with ResNet-18. It's kept here as a reference.

**Architecture:**
- 4 convolutional blocks with ReLU activation and MaxPooling
- BatchNorm on the 3rd conv layer
- Flatten → FC(7\*7\*128, 200) → ReLU → FC(200, 12)

> **Why not used?** Transfer learning with ResNet-18 achieves better accuracy with less training time.

In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  EXPERIMENTAL: Custom CNN from scratch (NOT USED in final model) │
# │  Kept as a reference for comparison with transfer learning.      │
# └──────────────────────────────────────────────────────────────────┘

# import torch.nn as nn
# class trash_classifier(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.conv=nn.Sequential(
#         nn.Conv2d(in_channels=3,out_channels=16,kernel_size=3),#Shape(128 -> 126) actual 224 X 224 X 3
#         nn.ReLU(),#Shape (222)
#         nn.MaxPool2d(2,2),#Shape (222 -> 111) so 222 X 222 X 16

#         nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,padding=1),#Shape (111 -> 111 ) as we use padding there , so 222 X 222 X 16.
#         nn.ReLU(),
#         nn.MaxPool2d(2,2), # shape (111 -> 55) so 55 X 55 X 32

#         nn.Conv2d(32, 64, 3, padding=1),
#         nn.BatchNorm2d(64),
#         nn.ReLU(),
#         nn.MaxPool2d(2,2),

#         nn.Conv2d(64,128, 3, padding=1),
#         nn.ReLU(),
#         nn.MaxPool2d(2,2),


#     )
#     self.fc=nn.Sequential(
#         nn.Flatten(),
#         nn.Linear(7*7*128,200),
#         nn.ReLU(),
#         nn.Linear(200,12)
#     )
#   def forward(self,x):
#       x=self.conv(x)
#       x=self.fc(x)
#       return x

---
## 8. Load Pre-trained ResNet-18 Model

**Transfer Learning** approach:
- Load ResNet-18 pre-trained on **ImageNet** (1.2M images, 1000 classes)
- Replace the final fully-connected layer: `512 features → 12 classes`
- The pre-trained CNN layers already know how to extract useful visual features (edges, textures, shapes)

```
ResNet-18 Architecture:
├── conv1 → bn1 → relu → maxpool     (frozen)
├── layer1 (2 BasicBlocks)             (frozen)
├── layer2 (2 BasicBlocks)             (frozen)
├── layer3 (2 BasicBlocks)             (frozen)
├── layer4 (2 BasicBlocks)             (frozen)
├── AdaptiveAvgPool2d → 512-dim vector
└── FC: 512 → 12                       (TRAINABLE)
```

In [10]:
# Load ResNet-18 with weights pre-trained on ImageNet
model=models.resnet18(pretrained=True)

# Replace the final FC layer to match our 12 trash categories
# Original: FC(512, 1000) for ImageNet → Modified: FC(512, 12) for trash
model.fc=nn.Linear(model.fc.in_features,12)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 146MB/s]


### 8.1 Freeze Feature Extractor Layers

**Why freeze?**
- The convolutional layers already extract powerful visual features from ImageNet training
- Freezing prevents their weights from being modified during training
- Only the final FC layer (classification head) will learn our 12 trash categories
- This is **faster** and **prevents overfitting** on smaller datasets

In [11]:
# Freeze ALL model parameters (convolutional + FC layers)
for param in model.parameters():
    param.requires_grad=False

# Unfreeze ONLY the final FC layer so it can learn during training
for param in model.fc.parameters():
    param.requires_grad=True

---
## 9. Define Loss Function and Optimizer

| Component | Choice | Reason |
|-----------|--------|--------|
| **Loss** | `CrossEntropyLoss` | Standard for multi-class classification (softmax + NLL) |
| **Optimizer** | `Adam` | Adaptive learning rate, works well out of the box |
| **Learning Rate** | `0.001` | Default for Adam, good starting point |
| **Weight Decay** | `1e-4` | L2 regularization to prevent overfitting |

In [12]:
# CrossEntropyLoss: combines LogSoftmax + NLLLoss in a single class
# Suitable for multi-class classification with C classes
loss_fn= nn.CrossEntropyLoss()

# Adam optimizer — only optimizes the FC layer parameters (since others are frozen)
# weight_decay adds L2 regularization penalty to prevent overfitting
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001,weight_decay=1e-4)

---
## 10. Training Loop

Train the FC layer for **5 epochs**. Each epoch:
1. Set model to **training mode** (`model.train()`)
2. Iterate over all batches in the training DataLoader
3. **Forward pass**: compute predictions
4. **Compute loss**: compare predictions vs true labels
5. **Backward pass**: compute gradients
6. **Update weights**: optimizer step
7. Log average loss and learning rate

In [26]:
for epoch in range(5):
  model.train()       # Set model to training mode (enables dropout, batchnorm updates)
  total_loss=0         # Accumulator for the total loss across all batches

  for (images,labels) in train_loader:
    # Forward pass: feed images through the model to get predictions
    outputs=model(images)

    # Calculate cross-entropy loss between predictions and true labels
    loss=loss_fn(outputs,labels)

    # Backward pass: compute gradients and update weights
    optimizer.zero_grad()   # Clear gradients from the previous batch
    loss.backward()         # Backpropagate the loss to compute gradients
    optimizer.step()        # Update FC layer weights using Adam optimizer

    total_loss+=loss.item() # Accumulate batch loss (detached from computation graph)

  # Log epoch summary: average loss and current learning rate
  current_lr=optimizer.param_groups[0]['lr']
  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}/5, Loss: {avg_loss:.4f}, Learning Rate = {current_lr:.6f}")

Epoch 1/5, Loss: 0.2699, Learning Rate = 0.001000
Epoch 2/5, Loss: 0.2608, Learning Rate = 0.001000
Epoch 3/5, Loss: 0.2505, Learning Rate = 0.001000
Epoch 4/5, Loss: 0.2488, Learning Rate = 0.001000
Epoch 5/5, Loss: 0.2361, Learning Rate = 0.001000


---
## 11. Evaluate on Training Set

Measure accuracy on the training set to check how well the model learned the training data. A large gap between train and validation accuracy would indicate **overfitting**.

In [27]:
train_correct = 0   # Counter for correctly classified images
train_total = 0     # Counter for total images evaluated

for images, labels in train_loader:
    outputs = model(images)
    _, preds = torch.max(outputs, 1)    # Get the index of the highest score (predicted class)

    train_correct += (preds == labels).sum().item()  # Count correct predictions
    train_total += labels.size(0)                    # Count total samples in this batch

# Calculate and print overall training accuracy
train_acc = train_correct / train_total
print(f"Train Accuracy: {train_acc * 100:.2f}%")

Train Accuracy: 92.34%


---
## 12. Evaluate on Validation Set

Validation accuracy tells us how well the model generalizes to unseen data. Using `torch.no_grad()` disables gradient computation for faster inference and lower memory usage.

In [28]:
model.eval()        # Set model to evaluation mode (disables dropout, freezes batchnorm)
correct=0           # Counter for correct predictions
total=0             # Counter for total predictions

with torch.no_grad():       # Disable gradient computation for efficiency
  for (images,labels) in val_loader:
    output=model(images)                        # Forward pass
    _,pred=torch.max(output,1)                  # Get predicted class index
    correct+=(pred==labels).sum().item()         # Count correct predictions
    total+=labels.size(0)                        # Count total samples

# Calculate and print validation accuracy
accuracy=100*correct/total
print(f"Total Accuracy : {accuracy:.2f}%")

Total Accuracy : 92.08%


---
## 13. Evaluate on Test Set

Final evaluation on the **held-out test set** — data the model has never seen during training or hyperparameter tuning. This gives the most unbiased estimate of real-world performance.

In [30]:
model.eval()        # Ensure model is in evaluation mode
correct=0
total=0

with torch.no_grad():   # No gradients needed for evaluation
    for (images,labels) in test_loader:
        outputs=model(images)                       # Forward pass
        _,pred=torch.max(outputs,1)                 # Get predicted class index
        correct+=(pred==labels).sum().item()         # Count correct predictions
        total+=labels.size(0)                        # Count total samples

# Calculate and print test accuracy
accuracy=100*correct/total
print(f"Total Accuracy : {accuracy:.2f}%")

Total Accuracy : 92.14%


---
## 14. Save the Trained Model

Save only the `state_dict` (model parameters/weights), not the entire model object.

**Why `state_dict` instead of the full model?**
- Smaller file size
- More portable across different PyTorch versions
- Gives flexibility to modify the architecture before loading weights

**To load later:**
```python
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(512, 12)
model.load_state_dict(torch.load('trash_classifier_resnet18.pth'))
model.eval()
```

In [31]:
# Save the trained model weights to a .pth file
# state_dict() contains all learnable parameters (weights and biases)
torch.save(model.state_dict(),'/kaggle/working/trash_classifier_resnet18.pth')
print("Model saved successfully!")

Model saved successfully!


---
## ✅ Training Complete — Summary

| Metric | Value |
|--------|-------|
| **Model** | ResNet-18 (transfer learning) |
| **Trainable Parameters** | FC layer only (512 × 12 + 12 = 6,156) |
| **Epochs** | 5 |
| **Final Training Loss** | 0.2361 |
| **Train Accuracy** | 92.34% |
| **Validation Accuracy** | 92.08% |
| **Test Accuracy** | 92.14% |
| **Saved Model** | `trash_classifier_resnet18.pth` |